# Heterodyne heatmap vs. cavity detuning, with a bright-mode fit (archived)

A single frame of the spectral heatmaps: the heterodyne spectrum $S_{het}(\omega)$ of a 4×1
system (equally spaced Ωx around 300 kHz, $g/2\pi = -40$ kHz, $\kappa/2\pi = 200$ kHz) as a
function of the cavity detuning Δ. For each Δ the lowest- and highest-frequency peaks are found,
and their splitting is fitted to $2\sqrt{s(\Delta - \Delta_0)^2 + G^2}$, the avoided crossing of the
cavity with the collective bright mode (expected minimum splitting $2\sqrt{N}|g|$ near $\Delta = -\Omega$).
Runs in about 10 s.

**Updated Nov 2026** to current levisim (`calculate_*_spectrum` → `*_spectrum`,
`setup_hybridisation_power_series` → `load_in_power_series`, the loop variable `ax` renamed
`axes` so it no longer shadows the plot axes). The fit reproduces the stored output of the original
run: stretch 0.3088, Δ₀ = −338.4 kHz, G = 34.98 kHz.

In [ ]:
import numpy as np
from scipy.signal import find_peaks
from scipy.optimize import curve_fit

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LogNorm

from levisim import *

set_plt_rcParams_to_xmgrace_style()
plt.rcParams['axes.xmargin'] = 0


def plot_heatmap(
    x, y, intensity,
    title='PSDs',
    ylabel='(kHz)',
    log_scale=False,
    **kwargs,
):
    if log_scale:
        norm = mcolors.LogNorm(vmin=np.max([intensity.min(), 1e-10]), vmax=intensity.max())
    else:
        norm = None

    fig, ax = plt.subplots(dpi=300)
    im = ax.pcolormesh(
        x, y, intensity,
        shading='nearest',
        norm=norm,
        # cmap='seismic',
        cmap='viridis',
    )
    ax.set_title(title)
    ax.set_xlabel('Frequency (kHz)')
    ax.set_ylabel(ylabel)
    ax.set_xlim(x.min(), x.max())
    ax.set_ylim(y.min(), y.max())

    # Add colorbar
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Intensity (log scale)" if log_scale else "Intensity")

    Ntot = kwargs.get('Ntot', 2)
    circle_params = kwargs.get('circle_params')
    if circle_params:
        x0, y0, r0 = circle_params
        # G_circle = plt.Circle((x0, y0), r0, color='magenta', alpha=0.5)
        # ax.add_patch(G_circle)

        # Add a double-headed arrow spanning the diameter
        ax.annotate(
            '',
            xy=(x0 - r0, y0),  # Start point (left side of circle)
            xytext=(x0 + r0, y0),  # End point (right side of circle)
            arrowprops=dict(
                arrowstyle='<->',  # Double-headed arrow
                color='white',  # Arrow color
                lw=0.5,  # Line width
                shrinkA=0,  # No shrinking at start
                shrinkB=0,  # No shrinking at end
            )
        )

        # Add white text just above the center of the circle
        ax.text(
            x0,
            y0 + r0 * 0.2,
            r'$2\sqrt{N}g=$' + f'{2*r0:.1f} kHz',
            color=white,
            ha='center',  # Horizontal alignment (centered)
            va='bottom',  # Vertical alignment (bottom of text at y-position)
            fontsize=8,  # Adjust font size as needed
            weight='bold'  # Optional: makes text bold
        )

        x_left = x[x < x0] - x0
        x_right = x[x > x0] - x0
        bright_mode_left = y0 + (r0**2 - x_left**2) / x_left
        bright_mode_right = y0 + (r0**2 - x_right**2) / x_right

        ax.plot(x_left + x0, bright_mode_left, color=white, ls='-', linewidth=0.5)
        ax.plot(x_right + x0, bright_mode_right, color=white, ls='-', linewidth=0.5)

    # for each y slice, find and plot the lowest and highest frequency peaks
    diffs = np.full(len(y), np.nan)
    smallest_ω_peaks_kHz = []
    for ii in range(len(y)):
        peaks, props = find_peaks(
            intensity[ii],
            # rel_height=0.5,
        )
        ω_peaks = sorted(ω_range[peaks])
        ω_peaks_kHz = np.array(ω_peaks) / kHz

        smallest_ω_peaks_kHz.append(ω_peaks_kHz[0])
        ax.plot(ω_peaks_kHz[0], y[ii], color=blue, marker='o', markersize=0.5)

        if len(ω_peaks) == Ntot + 1:
            ax.plot(ω_peaks_kHz[-1], y[ii], color=red, marker='o', markersize=0.5)
            diffs[ii] = ω_peaks_kHz[-1] - ω_peaks_kHz[0]


    fit_ranges = {
        2: (-160, -100),
        3: (-195, -110),
        4: (-265, -120),
        5: (-270, -155),
        6: (-300, -155),
        7: (-285, -165),
        8: (-350, -195),
    }
    y_min, y_max = fit_ranges[Ntot]
    y_min, y_max = -550, -150
    y_min, y_max = -795, -150
    mask = (y >= y_min) & (y <= y_max)
    y_subset = y[mask]
    diffs_subset = diffs[mask]

    # # Fit a parabola (degree=2) to the subset
    # coefficients = np.polyfit(y_subset, diffs_subset, 2)
    # a, b, c = coefficients
    # y_floor = -b / (2*a)
    # diffs_floor = c - b**2 / (4*a)
    # y_fit = np.linspace(y_floor - 200, y_floor + 200, 100)
    # diffs_fit = a * y_fit**2 + b * y_fit + c

    # Fit a model diff = |2√[(y-y0)²+|G|²]| curve
    nan_mask = ~np.isnan(diffs_subset) & ~np.isnan(y_subset)
    y_subset = y_subset[nan_mask]
    diffs_subset = diffs_subset[nan_mask]

    print(f'{y_subset=}')
    print(f'{diffs_subset=}')

    def diff_func(y, stretch, y0, G):
        return 2 * np.sqrt(stretch * (y - y0)**2 + G**2)

    initial_guess = [1, y0, r0]
    print(f'{initial_guess=}')

    visual_guess = [0.5, -340, 40]
    print(f'{visual_guess=}')

    bounds = (
        [0.1, 2.0*y0, 0.1*r0],
        [2.0, 0.5*y0, 4.0*r0],
    )

    print(f'{bounds[0]=}')
    print(f'{bounds[1]=}')

    params, covariance = curve_fit(
        diff_func,
        y_subset,
        diffs_subset,
        p0=initial_guess,
        bounds=bounds,
        method='trf',
        # method='dogbox',
    )
    print(f'{params=}')
    stretch_fit, y0_fit, G_fit  = params
    print(f'({stretch_fit=}')
    print(f'({y0_fit=} kHz')
    print(f'({G_fit=} kHz')

    y_floor, diffs_floor = y0_fit, 2 * G_fit
    y_fit = np.linspace(-750, -50, len(y))
    fit_guess = [stretch_fit, y0_fit, G_fit]

    fig2, ax2 = plt.subplots(dpi=300)
    ax2.plot(y, diffs, color=blue, marker='.')
    ax2.set_xlabel(ylabel)
    ax2.set_ylabel('Bright mode eigenvalue difference (kHz)')
    ax2.scatter(y_subset, diffs_subset, color=red, alpha=0.2, label="Subset to fit to")
    ax2.plot(y_fit, diff_func(y_fit, *fit_guess), color=red, ls='--', label="Fitted Diff. Curve")
    # ax2.plot(y_fit, diff_func(y_fit, *initial_guess), 'g--', label='Initial Guess')
    # ax2.plot(y_fit, diff_func(y_fit, *visual_guess), 'c--', label='Visual Guess')
    ax2.axhline(y=diffs_floor, color='k', ls='--', lw=0.5, label=f'min($\lambda_+ - \lambda_-$)={diffs_floor:.1f} kHz')
    ax2.axvline(x=y_floor, color='k', ls='--', lw=0.5, label=f'@ $\Delta$={y_floor:.1f} kHz')
    ax2.legend()

    # add line to other graph
    ax.axhline(y=y_floor, color=white, ls='--', lw=0.5)
    ax.plot(x, -x, color=white, ls='--', lw=0.5)

    projected_largest_ω_peaks_kHz = []
    for ii, peak in enumerate(smallest_ω_peaks_kHz):
        projected_largest_ω_peaks_kHz.append(peak + diff_func(y[ii], *fit_guess))

    print(f'{len(smallest_ω_peaks_kHz)=}')
    print(f'{len(projected_largest_ω_peaks_kHz)=}')
    ax.scatter(np.array(projected_largest_ω_peaks_kHz), y, color=green, marker='.')
    plt.show()


for N, axes in [
    # (2, 1),
    # (3, 1),
    (4, 1),
    # (5, 1),
    # (6, 1),

    # (1, 2),
    # (2, 2),
    # (3, 2),

    # (7, 1),
    # (8, 1),
]:
    Ntot = N * axes

    Ω0, g0 = 300, -40
    toy_params = {
        'Ω_model': 'equal spacing',
        'Ω_mid': Ω0 * kHz,
        'Ω_span': 5 * Ntot * kHz,

        # 'Ω_model': 'pairs',
        # 'Ω_first': 312*kHz,
        # 'Ω_interpair_gap': 15*kHz,
        # 'Ω_intrapair_gap': 1.5*kHz,

        # 'Ω_model': 'degenerate',
        # 'Ω0': Ω0 * kHz,

        'toy_gx': g0 * kHz,
        'toy_gy': g0 * kHz,

        'use_Γ_formula': True,
    }
    cavity_kwargs = {
        'Δ': -450*kHz,
        'κ': 200*kHz,
        # 'L': 0.96*cm,
        # 'Finesse': 100_000,
        'θhom': π/4,
    }
    gas_kwargs = {
        'P': 1e-7*mbar,
        'T': 300*K,
        'vth': 500,  # m/s
    }
    method = QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION

    low, high, low_res = 0, 800, 2.5
    hd_low, hd_high, hd_res = 296, 304, 0.01
    hd_low, hd_high, hd_res = Ω0 - 5 * Ntot, Ω0 + 5 * Ntot, 0.5
    ω_range = np.concatenate(
        (
            np.arange(low, hd_low, low_res) * kHz,
            np.arange(hd_low, hd_high, hd_res) * kHz,
            np.arange(hd_high, high, low_res) * kHz,
        ),
    )

    x_shape = len(ω_range)

    num_y = 375
    couplings = np.linspace(-2, -200, num_y, endpoint=False) * kHz
    detunings = -np.linspace(50, 800, num_y, endpoint=False) * kHz
    dampings = +np.linspace(5, 500, num_y, endpoint=False) * kHz

    step_y = 5
    detunings = -np.arange(50, 801, step_y) * kHz

    # DEFINE THE Y VARIABLE AND SPECTRUM TYPES HERE!!!!
    y_variables = dampings
    y_variables = couplings
    y_variables = detunings
    y_shape = len(y_variables)

    intensity = np.zeros(shape=(x_shape, y_shape))

    spec_type = 'psd'
    spec_type = 'hom'
    spec_type = 'het'
    index = 4

    psd, hom, het = spec_type == 'psd', spec_type == 'hom', spec_type == 'het'

    for ii, y_variable in enumerate(y_variables):

        if y_variables is couplings:
            toy_params['toy_gx'] = y_variable
            toy_params['toy_gy'] = y_variable
            ylabel = 'Couplings (kHz)'

        elif y_variables is detunings:
            cavity_kwargs['Δ'] = y_variable
            ylabel = 'Cavity Detuning (kHz)'

        elif y_variables is dampings:
            cavity_kwargs['κ'] = y_variable
            ylabel = 'Optical Damping (kHz)'

        mtr = MultiTrap(
            num_traps=N,
            Cartesian_axes_per_trap=axes,
            ω_range=ω_range,

            toy_params=toy_params,
            cavity_kwargs=cavity_kwargs,
            gas_kwargs=gas_kwargs,

            load_in_power_series=False,
        )

        if hom:
            intensity[:, ii] = mtr.homodyne_spectrum(method)
        elif het:
            intensity[:, ii] = mtr.heterodyne_spectrum(method)
        elif psd:
            intensity[:, ii] = mtr.all_position_power_spectra(method, zpf_scaled=False)[0][:, index]

    x, y, intensity = ω_range / kHz, y_variables / kHz, intensity.T

    index_to_spec = {
        0: '$S_{x_1 x_1}$',
        1: '$S_{y_1 y_1}$',
        2: '$S_{z_1 z_1}$',
        3: '$S_{x_2 x_2}$',
        4: '$S_{y_2 y_2}$',
        5: '$S_{z_2 z_2}$',
    }

    # print(f'{x.shape=}')
    # print(f'{y.shape=}')
    # print(f'{intensity.shape=}')

    spec_type_str = index_to_spec[index] if psd else '$S_{het}$'
    spec_type_str = spec_type_str + f' ({N}x{axes})'
    if psd or het:
        plot_heatmap(
            x, y, intensity,
            title=f'{spec_type_str}',
            ylabel=ylabel,
            log_scale=True,
            circle_params=(
                Ω0,
                -Ω0,
                np.abs(np.sqrt(Ntot) * g0),
            ),
            Ntot=Ntot,
        )

    else:
        x, y = np.meshgrid(x, y)
        S_hom = intensity

        # split into squeezed and non-squeezed
        squeezed_threshold = 1
        non_squeezed = np.where(S_hom >= squeezed_threshold, S_hom, np.nan)
        squeezed = np.where(S_hom < squeezed_threshold, S_hom, np.nan)

        fig, ax = plt.subplots()

        # plot non-squeezed values on a log scale
        pc_positive = ax.pcolormesh(
            x, y, non_squeezed,
            cmap='Wistia_r',
            norm=LogNorm(),
            shading='nearest',
        )

        plt.xlabel('Frequency (kHz)')
        spec_type_str = '$S_{hom}$'
        if y_variables is couplings:
            plt.title(f'{spec_type_str} vs. $g$ (photons/Hz)')
            plt.ylabel('Light-mech coupling (kHz)')

        elif y_variables is dampings:
            plt.title(f'{spec_type_str} vs. $κ$ (photons/Hz)')
            plt.ylabel('Optical damping (kHz)')

        elif y_variables is detunings:
            plt.title(f'{spec_type_str} vs. $Δ$ (photons/Hz)')
            plt.ylabel('Cavity detuning (kHz)')

        # [left, bottom, width, height]
        cax_positive = fig.add_axes([0.1, 0.999, 0.35, 0.03])
        cbar_positive = plt.colorbar(pc_positive, cax=cax_positive, orientation='horizontal')
        cbar_positive.ax.xaxis.set_label_position('top')
        cbar_positive.set_label('Non-Squeezed $S_{hom} ≥ 1$')

        # plot squeezed values on a linear scale
        pc_negative = ax.pcolormesh(
            x, y, squeezed,
            cmap='coolwarm',
            vmin=np.nanmin(squeezed),
            vmax=squeezed_threshold,
            shading='nearest',
        )

        cax_negative = fig.add_axes([0.5, 0.999, 0.35, 0.03])
        cbar_negative = plt.colorbar(pc_negative, cax=cax_negative, orientation='horizontal')
        cbar_negative.ax.xaxis.set_label_position('top')
        cbar_negative.set_label('Squeezed $S_{hom} < 1$')

        plt.show()

In [ ]:
mtr.info